In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyedfread import edf
import pickle
from glob import glob
from os.path import join
%matplotlib inline
import seaborn as sns
from PSPupil import Preprocessing as PS
from PSPupil import dataset as dt
import mne
import scipy

from scipy import stats, signal, interpolate
from scipy.stats import ttest_ind, linregress, f_oneway
from scipy.optimize import curve_fit
import svgutils.transform as sg
from decim.adjuvant import slurm_submit as slu
import matplotlib.colors
import itertools
import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp as mc


def smaller_than(value):
    logs=np.append(np.logspace(-15, 1, 17), .05)
    sorteds=np.sort(np.append(logs, value))
    index=np.where(sorteds==value)[0]
    return sorteds[index+1][0]

def func_powerlaw(x, m, c, c0):
    return c0 + x**m * c

def permutation_cluster_test(x1, x2):
    f, clusters, pvalues, h0=mne.stats.permutation_cluster_test([x1,x2],
                                                                     threshold=dict(start=0, step=0.2))
    return pvalues

def rgb_to_hex(r, g, b):
    return '#{:02x}{:02x}{:02x}'.format(r, g, b)


def smaller_than(value):
    logs=np.append(np.logspace(-15, 1, 17), .05)
    sorteds=np.sort(np.append(logs, value))
    index=np.where(sorteds==value)[0]
    return sorteds[index+1][0]

def p_annot(p_value):
    if p_value < 0.001:
        text = '<{0}'.format(smaller_than(p_value))
    else:
        text = '%.3f' % p_value
        
def p_annot(p_value, dec):
    10**-3
    if p_value < 10**-dec:
        text = '<{0}'.format(smaller_than(p_value))
    else:
        text = '{}'.format(np.round(p_value, dec))
    return text

In [9]:
edition = '/Volumes/psp_data/PSP/Pupil_Preprocessed_2024-06-17/'
condition = 'bp_non_normalized'

## Get Data

In [10]:
pat_data = pd.read_csv('/Users/kenohagena/Documents/Forschung/PSP/data/meta/Patientencharakteristika_Auswertung_Pupillometrie_Marc.csv', 
                       delimiter=';')
pat_data = pat_data.rename(columns={'ID ProPSP': 'subject',
                        'Alter Erstuntersuchung': 'age',
                        'Erkrankungsdauer (Jahre)': 'disease_duration',
                        'PSP Rating Scale': 'psp_rating_scale'}).drop('Patientennummer Pupillometrie', axis=1)

pat_data.disease_duration = pat_data.disease_duration.str.replace(',','.')


auc = pd.read_hdf(join(edition, 'PowerFrequencies', 'auc_bp_non_normalized.hdf'), key='data')
psp_baseline_auc = auc.loc[(auc.group == 'PSP')&(auc.session == 'Baseline')]
psp_baseline_auc = psp_baseline_auc.drop('condition', axis=1)
cov = pd.read_hdf(join(edition, 'PowerFrequencies', 'CoV.hdf'), key='data')
cov = cov.loc[cov.condition == 'bp_non_normalized']
cov = cov.groupby(['group', 'session', 'subject']).mean().reset_index()
cov['cov']= cov.loc[:, 'std'] / cov.loc[:, 'mean']
fooof = pd.read_hdf(join(edition, 'SPECPARAMS', 'Specparams_2024-06-18.hdf'), key= 'data')


### Exclude PD subject 12
cov = cov.set_index(['group', 'subject']).drop(('PD', '12')).reset_index()
fooof = fooof.set_index(['group', 'subject']).drop(('PD', '12')).reset_index()
auc = auc.set_index(['group', 'subject']).drop(('PD', '12')).reset_index()

In [11]:
corr_data= pd.concat([pat_data.set_index(pat_data.subject.astype(int)), 
                      psp_baseline_auc.set_index(psp_baseline_auc.subject.astype(int)),
                     cov.loc[(cov.group == 'PSP')&(cov.session == 'Baseline')].\
          set_index(cov.loc[(cov.group == 'PSP')&(cov.session == 'Baseline')].subject.astype(int)),
                     fooof.loc[(fooof.group == 'PSP')&(fooof.session == 'Baseline')].\
                      set_index(fooof.loc[(fooof.group == 'PSP')&(fooof.session == 'Baseline')].subject.astype(int))], axis=1)

In [12]:
gait = pd.read_table('/Volumes/psp_data/PSP/data/raw/GaitRite/GAITRite_data_2.tsv', header=[0, 1], index_col=[0, 1, 2], decimal=",")

cols = ['Velocity (cm/sec) normal', 'Step Length (cm) mean normal', 
        'Cadence (Steps/Min) normal', 'Single Support (%GC) mean normal','Step Length SymmetryRatio normal',
       'Velocity (cm/sec) fast', 'Step Length (cm) mean fast', 
        'Cadence (Steps/Min) fast', 'Single Support (%GC) mean fast','Step Length SymmetryRatio fast',
       'Velocity (cm/sec) turn', 'Step Length (cm) mean turn', 
        'Cadence (Steps/Min) turn', 'Single Support (%GC) mean turn','Step Length SymmetryRatio turn']

cols_more = ['Velocity (cm/sec) normal', 'Cadence (Steps/Min) normal', 'Step Length (cm) mean normal', 'Cycle Time (sec) mean normal', 'Single Support (%GC) mean normal', 'Double Support (%GC) mean normal', 'Swing (%GC) mean normal', 'Step Length SymmetryRatio normal',
             'Velocity (cm/sec) fast', 'Cadence (Steps/Min) fast', 'Step Length (cm) mean fast', 'Cycle Time (sec) mean fast', 'Single Support (%GC) mean fast', 'Double Support (%GC) mean fast', 'Swing (%GC) mean fast', 'Step Length SymmetryRatio fast',
             'Velocity (cm/sec) turn', 'Cadence (Steps/Min) turn', 'Step Length (cm) mean turn', 'Cycle Time (sec) mean turn', 'Single Support (%GC) mean turn', 'Double Support (%GC) mean turn', 'Swing (%GC) mean turn', 'Step Length SymmetryRatio turn']
             
       
             

gait_reduced = gait.copy()
gait_reduced.columns = gait_reduced.columns.droplevel(0)

gait_reduced = gait_reduced.reset_index().set_index('level_0').loc[:, cols_more]



c = corr_data.copy()
c.index = c.index.astype(str)
gait_corr = pd.concat([gait_reduced, c], axis=1)#.loc[corr_data.index.astype(str)]



# invalid data, set to NaN

gait_corr.loc['2', 'Step Length SymmetryRatio turn'] = np.nan


# Exclude subjects with no gait data

gait_corr = gait_corr.dropna(axis=1, how='all').loc[~gait_corr.loc[:, 'Velocity (cm/sec) normal'].isnull()]


/Users/kenohagena/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:26: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.



In [13]:
# New dataframe with subjects that have gait data (n=15)
# create format with columns for gait parameter + extra column for condition ('fast', 'normal', 'turn')

corr_data["RS"] = corr_data['Subtypen']
corr_data.RS = corr_data.RS.str.contains('PSP-RS')


clinical_df = corr_data.drop('subject', axis=1).reset_index()
clinical_df.subject = clinical_df.subject.astype(str)
clinical_df.disease_duration = clinical_df.disease_duration.astype(float)

cols = []

normal = []
fast = []
turn = []

c = gait_reduced.columns
for column in c:
    d = gait_reduced.loc[:, column]
    column_names = column.split()
    column_names = [i for i in column_names if '(' not in i]
    condition = [column_names[-1]]*len(d)
    category = ''.join(column_names[:-1])
    d = pd.DataFrame({category: d.values, 'condition':condition, 'index':d.index}).set_index(['index', 'condition'])
    if column_names[-1] == 'turn':
        turn.append(d)
    if column_names[-1] == 'normal':
        normal.append(d)
    if column_names[-1] == 'fast':
        fast.append(d)
        


normal = pd.concat(normal, axis=1).reset_index().rename(columns = {'index':'subject'})

cols = normal.columns
normal = normal.merge(clinical_df, on='subject', how='outer')
turn = pd.concat(turn, axis=1).reset_index().rename(columns = {'index':'subject'}).merge(clinical_df, on='subject', how='outer')
fast = pd.concat(fast, axis=1).reset_index().rename(columns = {'index':'subject'}).merge(clinical_df, on='subject', how='outer')


gcp = pd.concat([normal, 
                 fast, 
                 turn], axis=0).reset_index()

gcp = gcp.drop('group', axis=1)
gcp.loc[gcp.RS == True, 'group'] = 'PSP-RS'
gcp.loc[gcp.RS == False, 'group'] = 'vPSP'
gcp.loc[gcp.subject.str[0] == 'I', 'group'] = 'PD'
gcp.loc[gcp.subject.str[0] == 'C', 'group'] = 'HC'


# invalid data, set to NaN

gcp.loc[(gcp.subject == '2')&(gcp.condition == 'turn'), 'StepLengthSymmetryRatio'] = np.nan

In [14]:
label_dict = {'disease_duration': 'Disease duration (years)', 
     'psp_rating_scale': 'PSP Rating Scale', 
        'MoCA': 'MoCA', 
     'age': 'Age (years)', 
     'auc': 'AUC', 
     'CoV': 'CoV', 
     'frequency': 'Frequency (Hz)', 
     'offset': 'Hippus power',
    'StepLengthSymmetryRatio':"Step length AI",
    'StepLengthmean': 'Step length',
    'DoubleSupportmean': 'Double support phase',
    'SingleSupportmean': 'Single support phase',
     'Velocity': 'Velocity',
    'Cadence': 'Cadence',
    'MoCA': 'MoCA',
    'auc': 'AUC', 
     'cov': 'CoV', 
     'peak1_CF': 'Frequency (Hz)', 
     'peak1_PW': 'Hippus power',
    'aperiodic_offset': 'Offset',
    'aperiodic_exp': 'Exponent',
             'CycleTimemean': 'Step cycle time'}

## Mixed ANOVA

In [15]:
cols = ['Velocity', 'Cadence', 'StepLengthmean',
       'CycleTimemean', 'SingleSupportmean', 'DoubleSupportmean', 'Swingmean', 'StepLengthSymmetryRatio']
data = gcp.loc[~gcp.Velocity.isnull()]

In [16]:
gait_export = data.loc[:, cols+['group', 'RS', 'condition' ,  'subject']]
gait_export['group'] = gait_export['group'].replace({'PSP-RS': 'PSP',
                                                    'vPSP': 'PSP'})
gait_export.to_csv('GaitData_for_Pingouin.csv')

In [ ]:
#Folgenden Code in Google Colab laufen gelassen:

'''
import pandas as pd
import pingouin as pg
import numpy as np

from google.colab import files
uploaded = files.upload()

data = pd.read_csv('/content/GaitData_for_Pingouin.csv', index_col=0)
cols = ['Velocity', 'Cadence', 'StepLengthmean',
       'CycleTimemean', 'SingleSupportmean', 'DoubleSupportmean', 'Swingmean', 'StepLengthSymmetryRatio']
psp_data= data.loc[~data.RS.isnull()]       

for col in cols:
  aov = pg.mixed_anova(data=data, dv=col, between='group', within='condition',
                     subject='subject', correction=True, effsize="np2")
  posthocs = pg.pairwise_tests(data=data, dv='Velocity', between='group', within='condition',
                             subject='subject', interaction=True, padjust='bonf')
  rs_aov = pg.mixed_anova(data=psp_data, dv='Velocity', between='RS', within='condition',
                     subject='subject', correction=True, effsize="np2")
  rs_posthocs = pg.pairwise_tests(data=psp_data, dv='Velocity', between='RS', within='condition',
                             subject='subject', interaction=True, padjust='bonf')
  
  for table, label in zip([aov, posthocs, rs_aov, rs_posthocs], ['aov', 'posthocs', 'rs_aov', 'rs_posthocs']):
    table.to_csv(f'{col}_{label}.csv')
    files.download(f'{col}_{label}.csv')

'''

### Mixed ANOVA PSP vs. HC vs. PD

In [18]:
mean = gait_export.groupby(['condition', 'group']).mean().loc[:, cols].T
std = gait_export.groupby(['condition', 'group']).std().loc[:, cols].T

In [19]:
for cond in ['fast', 'normal', 'turn']:
    for param in mean.index:
        # load bomnferroni corrected post hoc pairwise t/test (pingouin module ion colab)
        posthoc= pd.read_csv('data/MixedAnovaGait_01-08-2025/{}_posthocs.csv'.format(param), index_col=0) 
        
        for combo in itertools.combinations(gait_export.group.unique(), 2): 
            try:
                mean.loc[param, (cond, 'posthoc_{0}_{1}'.format(combo[0], combo[1]))] =\
                posthoc.loc[(posthoc.condition == cond)&(posthoc.A == combo[0])&(posthoc.B == combo[1]), 'p-corr'].values
                
            except ValueError:
                mean.loc[param, (cond, 'posthoc_{0}_{1}'.format(combo[0], combo[1]))] =\
                posthoc.loc[(posthoc.condition == cond)&(posthoc.A == combo[1])&(posthoc.B == combo[0]), 'p-corr'].values
                

In [20]:
stacked = mean.stack('condition')

for i in std.stack('condition').index:
    for group in ['HC', 'PSP', 'PD']:
        if i[0] == 'StepLengthSymmetryRatio':
            
            stacked.loc[i, group] = '{0} ± {1}'.format(np.round(stacked.loc[i, group], decimals=2), np.round(std.stack('condition').loc[i, group], decimals=2))
        else:
            stacked.loc[i, group] = '{0} ± {1}'.format(np.round(stacked.loc[i, group], decimals=1), np.round(std.stack('condition').loc[i, group], decimals=1)) 
            
    for c in ['posthoc_PSP_HC', 'posthoc_PSP_PD', 'posthoc_HC_PD']:        
        stacked.loc[i, c] = p_annot(stacked.loc[i, c], dec=3) 

cols = ['PSP', 'PD', 'HC', 'posthoc_PSP_HC', 'posthoc_PSP_PD', 'posthoc_HC_PD']
stacked.loc[:, cols].astype(str).to_csv(join(edition, 'Figures', 'data', 'GAIT_table.csv'), sep = ',')

stacked= stacked.loc[:, cols]

stacked

group                                       PSP            PD            HC  \
                        condition                                             
Velocity                fast       113.0 ± 32.9  144.5 ± 36.7  204.8 ± 29.9   
                        normal      82.3 ± 24.6  100.5 ± 27.3  139.0 ± 15.1   
                        turn        79.5 ± 28.6   80.0 ± 24.6   98.5 ± 22.6   
Cadence                 fast       109.3 ± 17.0  132.0 ± 20.4  141.8 ± 17.0   
                        normal      93.4 ± 12.7  109.1 ± 12.9   110.9 ± 6.8   
                        turn        93.1 ± 13.6  107.2 ± 24.5  111.6 ± 12.6   
StepLengthmean          fast        61.5 ± 13.0   66.1 ± 15.1    86.7 ± 7.0   
                        normal      52.0 ± 11.1   55.0 ± 12.6    75.1 ± 5.8   
                        turn        48.9 ± 14.4   46.6 ± 14.7   52.0 ± 10.2   
CycleTimemean           fast          1.1 ± 0.2     0.9 ± 0.1     0.9 ± 0.1   
                        normal        1.3 ± 0.2     1.1 ± 0.1     1.1 ± 0.1   
                        turn          1.3 ± 0.2     1.2 ± 0.4     1.1 ± 0.1   
SingleSupportmean       fast         35.5 ± 2.3    37.4 ± 3.3    40.8 ± 2.6   
                        normal       33.6 ± 2.5    34.4 ± 2.2    37.7 ± 1.6   
                        turn         33.3 ± 3.3    33.3 ± 7.1    36.3 ± 2.9   
DoubleSupportmean       fast         28.7 ± 4.5    25.5 ± 6.7    17.0 ± 5.5   
                        normal       32.8 ± 4.9    31.3 ± 4.5    24.1 ± 3.1   
                        turn         34.6 ± 7.2   39.0 ± 11.5    32.0 ± 7.3   
Swingmean               fast         35.5 ± 2.3    37.4 ± 3.3    40.8 ± 2.6   
                        normal       33.6 ± 2.5    34.4 ± 2.3    37.7 ± 1.6   
                        turn         33.6 ± 2.4    32.7 ± 5.1    36.3 ± 2.9   
StepLengthSymmetryRatio fast        1.06 ± 0.06   1.06 ± 0.08   1.03 ± 0.03   
                        normal      1.09 ± 0.08   1.07 ± 0.08   1.03 ± 0.02   
                        turn        1.09 ± 0.06   1.11 ± 0.13   1.14 ± 0.13   

group                             posthoc_PSP_HC posthoc_PSP_PD posthoc_HC_PD  
                        condition                                              
Velocity                fast              <1e-07          0.178        <0.001  
                        normal            <1e-06          0.586        <0.001  
                        turn               0.443            1.0         0.323  
Cadence                 fast             <0.0001          0.023           1.0  
                        normal            <0.001           0.02           1.0  
                        turn               0.004          0.544           1.0  
StepLengthmean          fast             <0.0001            1.0         0.001  
                        normal            <1e-05            1.0        <0.001  
                        turn                 1.0            1.0           1.0  
CycleTimemean           fast              <0.001          0.037         0.974  
                        normal             0.003          0.042           1.0  
                        turn               0.033            1.0           1.0  
SingleSupportmean       fast              <1e-05          0.714         0.028  
                        normal            <0.001            1.0        <0.001  
                        turn               0.093            1.0           1.0  
DoubleSupportmean       fast              <1e-05            1.0         0.005  
                        normal           <0.0001            1.0        <0.001  
                        turn                 1.0            1.0         0.511  
Swingmean               fast              <1e-05          0.708         0.029  
                        normal            <0.001            1.0        <0.001  
                        turn                0.06            1.0         0.214  
StepLengthSymmetryRatio fast                 1.0            1.0           1.0  
                        n

In [21]:
mean_ANOVA_results= mean.copy()
for param in mean_ANOVA_results.index:
    aov = pd.read_csv('data/MixedAnovaGait_01-08-2025/{}_aov.csv'.format(param), index_col=0) 
    for term in ['group', 'condition', 'Interaction']:
        mean_ANOVA_results.loc[param, (term, 'F')] = aov.loc[(aov.Source == term), 'F'].values
        mean_ANOVA_results.loc[param, (term, 'p-value')] = aov.loc[(aov.Source == term), 'p-unc'].values #uncorrected p
        
mean_ANOVA_results = mean_ANOVA_results.drop(['fast', 'normal', 'turn'], axis=1)

mean_ANOVA_results = mean_ANOVA_results.applymap(lambda x: p_annot(x, dec=3))

/Users/kenohagena/anaconda3/lib/python3.6/site-packages/pandas/core/generic.py:3946: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  new_axis = axis.drop(labels, errors=errors)


In [22]:
mean_ANOVA_results.reset_index()

condition                    index   group          condition           \
group                                    F  p-value         F  p-value   
0                         Velocity  22.032   <1e-06   271.316   <1e-15   
1                          Cadence  12.254  <0.0001    83.606   <1e-15   
2                   StepLengthmean    12.1  <0.0001   202.398   <1e-15   
3                    CycleTimemean   7.934    0.001    44.004   <1e-13   
4                SingleSupportmean  11.009   <0.001     24.51   <1e-08   
5                DoubleSupportmean  11.839  <0.0001    67.224   <1e-15   
6                        Swingmean  12.555  <0.0001    50.787   <1e-14   
7          StepLengthSymmetryRatio   0.234    0.792    13.082  <0.0001   

condition Interaction          
group               F p-value  
0              24.366  <1e-12  
1               2.676   0.037  
2              21.224  <1e-11  
3               0.916   0.458  
4                0.83   0.509  
5               4.147   0.004  
6               2.766   0.032  
7               2.779   0.032

In [6]:
pd.read_csv('data/MixedAnovaGait_09-09-2025/Velocity_aov.csv')

,Source,SS,DF1,DF2,MS,F,p-unc,p-GG-corr,np2,eps,sphericity,W-spher,p-spher
0,group,79659.191654,2,44,39829.595827,22.032253,2.346060e-07,NaN,0.500366,NaN,NaN,NaN,NaN
1,condition,119505.885605,2,88,59752.942802,271.316428,2.328453e-38,1.428463e-19,0.860458,0.668366,False,0.503815,2.000279e-07
2,Interaction,21464.730607,4,88,5366.182652,24.365888,1.368624e-13,NaN,0.525513,NaN,NaN,NaN,NaN


In [7]:
pd.read_csv('data/MixedAnovaGait_09-09-2025/ALL_posthocs_globalBonf.csv')

,Contrast,condition,A,B,Paired,Parametric,T,dof,alternative,p-unc,BF10,hedges,DV,p-bonf-global,alpha_bonf_global,sig_bonf_global,sig_from_punc_global
0,group,-,HC,PSP,False,True,6.528475,24.728572,two-sided,8.150197e-07,33390.000,2.304459,Velocity,5.216126e-05,0.000781,True,True
1,group,-,PD,PSP,False,True,1.692844,28.000000,two-sided,1.015848e-01,0.995,0.601433,Velocity,1.000000e+00,0.000781,False,False
2,condition * group,fast,HC,PSP,False,True,8.224649,28.573738,two-sided,5.099180e-09,2101000.000,2.857521,Velocity,3.263475e-07,0.000781,True,True
3,condition * group,fast,PD,PSP,False,True,2.471969,28.000000,two-sided,1.978074e-02,3.071,0.878240,Velocity,1.000000e+00,0.000781,False,False
4,condition * group,normal,HC,PSP,False,True,7.739327,22.722273,two-sided,8.183336e-08,656800.000,2.751328,Velocity,5.237335e-06,0.000781,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,condition * group,fast,PD,PSP,False,True,0.280883,25.709657,two-sided,7.810479e-01,0.359,0.100397,StepLengthSymmetryRatio,1.000000e+00,0.000781,False,False
60,condition * group,normal,HC,PSP,False,True,-2.626895,14.597365,two-sided,1.937935e-02,4.027,-1.006963,StepLengthSymmetryRatio,1.000000e+00,0.000781,False,False
61,condition * group,normal,PD,PSP,False,True,-0.796145,26.423564,two-sided,4.330428e-01,0.443,-0.288348,StepLengthSymmetryRatio,1.000000e+00,0.000781,False,False
62,condition * group,turn,HC,PSP,False,True,1.373073,23.831656,two-sided,1.825120e-01,0.693,0.453193,StepLengthSymmetryRatio,1.000000e+00,0.000781,False,False
